# Comp universe and facets

Steps 1-3 of the mentor's recipe for getting comps:

1. **Data prep:** a universe of candidate comps covering the list's 2-digit SIC codes, HQ in the US.
2. **Multiple range:** drop firms, targets and comps alike, whose EV/EBITDA is outside 3-30.
3. **Facets:** descriptive attributes, available for every company, that together give a reasonable sense of its business model.

**Decisions this notebook follows (2026-10-24)**
- The universe is the 560-company list itself: each company is a target in turn, and its candidate comps are the others. `v0/universe.py` takes any list with the same columns, so a wider universe can be dropped in later.
- The list is the required snapshot (`company_list_v3`), already cleaned in earlier iterations: SIC codes zero-padded (BrightView 0700), investor-relations websites replaced. Nothing is re-done here.
- Financial columns are split off as soon as the list is read. EV/EBITDA is used only for the 3-30 range.
- Facets are filled by an LLM (Claude Haiku 5.5 through the Anthropic API, chosen 2026-10-24; `v0/fill_facets.py`, run outside the notebook) from what it knows about the company, given its name and website. The pages already saved by the website scrape only fill gaps: a company left with an unknown facet gets a second call with its pages, and only the unknown facets are taken from it (no new scraping). Every facet records its source (model knowledge, website or both). Nothing in the facet table is computed, and the model never sees financial fields.
- The model is a run option (`fill_facets.py --backend`, `--model`; Codex CLI is the other backend). Every company in a run is filled by the same model, and a profile made with another model is redone.
- Correctness is not checked yet: that is the manual review in the recipe's evals section.

Outputs: `outputs-local/comps/universe/<run time>/` (written here) and `outputs-local/comps/facets/` (written by `fill_facets.py`, read here).

In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# repo root = the folder holding data_manifest.json (works from the repo root, v0/ or v0/notebooks/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_manifest.json").exists())
sys.path.insert(0, str(ROOT / "v0"))
import facets as fc                     # also puts website_scrape/ and src/ on the path
import universe as uv
from fill_facets import DEFAULT_OUT as FACETS_OUT, SCRAPE_PAGE_TEXT, output_paths
from project_data import OUTPUTS_LOCAL, load_manifest, sha256, snapshot_path

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

RUN = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = OUTPUTS_LOCAL / "comps" / "universe" / RUN
manifest = load_manifest()
snap = manifest["required_snapshot"]
print("snapshot:", snap, "|", manifest["snapshots"][snap]["description"])
print("sha256:", sha256(snapshot_path()))

snapshot: company_list_v3 | v2 plus website fixes: investor-relations hosts replaced by main domains (Avista, Black Hills, Herc, Wingstop); Smith Douglas website supplied (was '0').
sha256: 3ab9a43e7426c1f41a77055a6d3ca9553845cb3046a954e28dc5c2a82c934eae


## 1. Candidate comps: SIC 2-digit codes and US HQ

The list is read as two tables keyed by `IQ_COMPANY_ID`: company information, and financials. A company is a candidate if Capital IQ gives its country as the US and its 2-digit SIC code is one of the list's codes. With the list as its own universe, every company passes by construction; the check matters once a wider universe is used.

In [2]:
info, financials = uv.load_list()
candidates = uv.select_candidates(info)
print(f"{len(info)} companies, {info['target_sic2_code'].nunique()} SIC 2-digit codes")
print(candidates[["us_hq", "sic2_in_target_list", "candidate"]].sum().to_string())

560 companies, 43 SIC 2-digit codes
us_hq                  560
sic2_in_target_list    560
candidate              560


Companies per 2-digit SIC code. Where a code has very few companies, a same-SIC comp may not exist in this universe. That isn't a reason to drop anything: SIC is a check, and the facets decide what counts as the same business.

In [3]:
sic2 = (candidates.groupby("target_sic2_code")
        .agg(companies=("IQ_COMPANY_ID", "size"),
             example_industries=("target_primary_sic_industry",
                                 lambda s: "; ".join(s.value_counts().index[:2])))
        .sort_values("companies", ascending=False))
print("codes with 1-2 companies:", (sic2["companies"] <= 2).sum())
sic2

codes with 1-2 companies: 8


,companies,example_industries
target_sic2_code,,
73,117,"Computer Programming, Data Processing, And Other Computer Related Services; Prepackaged Software"
28,52,"Pharmaceutical Preparations; Biological Products, Except Diagnostic Substances"
35,38,"Special Industry Machinery, not elsewhere classified; Farm Machinery and Equipment"
38,35,"Orthopedic, Prosthetic, and Surgical Appliances and Supplies; Electromedical and Electrotherapeutic Apparatus"
87,22,"Management Consulting Services; Engineering, Accounting, Research, Management, and Related Services"
79,18,Miscellaneous Amusement And Recreation; Amusement and Recreation Services
20,17,Bottled and Canned Soft Drinks and Carbonated Waters; Sugar And Confectionery Products
50,16,"Machinery, Equipment, And Supplies; Medical, Dental, and Hospital Equipment and Supplies"
37,16,Motor Vehicle Parts and Accessories; Aircraft


## 2. EV/EBITDA range: 3-30

Firms outside 3-30 (inclusive) are dropped from both sides: they are neither targets nor comps. The recipe asks for financials as of year end 2025. The list's multiples are used as given; **their as-of date is not documented** (an open question in PROJECT_STATE.md).

In [4]:
check = uv.multiple_check(financials)
print(check["drop_reason"].replace("", "kept").value_counts().to_string())
print("\nEV/EBITDA, all 560:")
print(check["ev_to_ebitda"].describe().round(2).to_string())

drop_reason
kept        533
above_30     20
below_3       7

EV/EBITDA, all 560:
count    560.00
mean      12.50
std        7.40
min        2.16
25%        7.22
50%       10.75
75%       15.49
max       45.72


In [5]:
dropped = (check[~check["kept"]]
           .merge(info[["IQ_COMPANY_ID", "target_company_name", "target_ciq_ticker",
                        "target_sic4_code", "target_primary_sic_industry"]], on="IQ_COMPANY_ID")
           .sort_values("ev_to_ebitda"))
dropped[["target_company_name", "target_ciq_ticker", "target_sic4_code",
         "target_primary_sic_industry", "ev_to_ebitda", "drop_reason"]].round(2)

,target_company_name,target_ciq_ticker,target_sic4_code,target_primary_sic_industry,ev_to_ebitda,drop_reason
4,"Chegg, Inc.",CHGG,7370,"Computer Programming, Data Processing, And Other Computer Related Services",2.16,below_3
19,TGS ASA,TGS,1382,Oil and Gas Field Exploration Services,2.45,below_3
2,"Atara Biotherapeutics, Inc.",ATRA,2836,"Biological Products, Except Diagnostic Substances",2.48,below_3
23,Unit Corporation,UNTC,1381,Drilling Oil and Gas Wells,2.49,below_3
8,"Forrester Research, Inc.",FORR,8700,"Engineering, Accounting, Research, Management, and Related Services",2.51,below_3
1,"American Integrity Insurance Group, Inc.",AII,6331,"Fire, Marine, and Casualty Insurance",2.62,below_3
16,"SIGA Technologies, Inc.",SIGA,2834,Pharmaceutical Preparations,2.91,below_3
6,"FIGS, Inc.",FIGS,2300,Apparel and Other Finished Products Made From Fabrics and Similar Materials,30.04,above_30
3,"Celsius Holdings, Inc.",CELH,2086,Bottled and Canned Soft Drinks and Carbonated Waters,30.64,above_30
0,"AAON, Inc.",AAON,3585,Air-Conditioning and Warm Air Heating Equipment and Commercial and Industrial Refrigeration Equipment,31.46,above_30


Which SIC codes lose companies, and the universe that remains.

In [6]:
companies, financials = uv.build_universe()
universe = companies[companies["in_universe"]]
lost = (companies.groupby("target_sic2_code")
        .agg(before=("IQ_COMPANY_ID", "size"), after=("in_universe", "sum")))
lost["dropped"] = lost["before"] - lost["after"]
print(f"universe: {len(universe)} companies, {universe['target_sic2_code'].nunique()} SIC 2-digit codes "
      f"(all {companies['target_sic2_code'].nunique()} still covered: "
      f"{set(universe['target_sic2_code']) == set(companies['target_sic2_code'])})")
lost[lost["dropped"] > 0].sort_values("dropped", ascending=False)

universe: 533 companies, 43 SIC 2-digit codes (all 43 still covered: True)


,before,after,dropped
target_sic2_code,,,
28,52,46,6
73,117,113,4
38,35,32,3
87,22,20,2
36,10,8,2
13,5,3,2
35,38,37,1
23,7,6,1
20,17,16,1


In [7]:
# Saved: the company table with every flag, the universe alone (no financials), and financials in their own table
OUT.mkdir(parents=True, exist_ok=True)
companies.to_csv(OUT / "companies.csv", index=False)
universe.drop(columns=["us_hq", "sic2_in_target_list", "candidate", "kept_multiple", "drop_reason", "in_universe"]
              ).to_csv(OUT / "universe.csv", index=False)
financials.to_csv(OUT / "financials.csv", index=False)
(OUT / "run.json").write_text(json.dumps({
    "run": RUN, "snapshot": snap, "snapshot_sha256": sha256(snapshot_path()),
    "ev_to_ebitda_range": uv.EV_EBITDA_RANGE, "companies": len(companies),
    "in_universe": int(companies["in_universe"].sum()),
    "dropped": companies.loc[~companies["kept_multiple"], "drop_reason"].value_counts().to_dict(),
}, indent=1), encoding="utf-8")
print("wrote", OUT.relative_to(ROOT))

wrote outputs-local\comps\universe\20261025T060358Z


## 3. Facets

Twelve descriptive facets per company (agreed 2026-10-24), filled by an LLM and checked by code. 3.1 defines them, 3.2 says which steps the model does and which code does, 3.3 shows the filled facets. How the fills were tested, and what each model costs, is in `facet_fill_runs.ipynb`.

### 3.1 The twelve facets

Defined once in `v0/facets.py`; the prompt, the answer schema and the checks are all built from that definition.

- **Text facets** get a short value written like the examples.
- **Category facets** get a category from a fixed list plus a short detail (`B2B: mechanical contractors, building owners`), so companies can be compared on the category without losing the detail. `one of` takes exactly one category, `any of` every one that applies.
- **One profile per company** (decided 2026-10-25). A company that runs several lines or businesses is described as one, covering all of them in each facet, largest first; the answer schema has room for one business only. Splitting into unrelated businesses is switched off for now (`facets.SPLIT_BUSINESSES`).
- **Status:** `filled`, `not_applicable` (products or services a company doesn't sell; counts as filled) or `unknown` (a last resort, when the model knows nothing that applies).
- **Company size is a category with no detail:** small, mid-sized, large or among the largest, within the company's own industry. With free text the model wrote "five thousand employees" even when digits were banned; a fixed list leaves no room for a figure. It is scale only: where the company operates is geography served, and how it sells is distribution channels. No facet gives financial information.

In [8]:
pd.DataFrame([{"ID": fc.FACET_ID[f.name], "Facet": f.label, "What it captures": f.captures, "Example values": f.examples,
               "Categories": f"{f.kind}: {', '.join(f.categories)}" if f.categories else ""}
              for f in fc.FACETS]).set_index("ID")

,Facet,What it captures,Example values,Categories
ID,,,,
F1,Core offering,"What the company sells, and to whom",Designs and manufactures rooftop HVAC units for commercial buildings,
F2,Primary products,The main products it sells,HR software; medical devices; hot-mix asphalt,
F3,Primary services,The main services it provides,consulting; freight logistics; payroll processing,
F4,Customer segment,Who buys its products or services,"B2B: mechanical contractors, commercial building owners; B2G: school districts","any of: B2B, B2C, B2G"
F5,End markets,The industries it serves,healthcare; retail; manufacturing,
F6,Revenue model,How it earns money,product sales: equipment sold through dealers; service fees: installation and repair,"any of: product sales, subscription, transaction or usage fees, licensing or royalties, service fees, project or con..."
F7,Value chain position,Its role in the industry,"OEM / manufacturer: builds HVAC units from purchased components, sold to contractors","any of: raw materials, component supplier, OEM / manufacturer, brand owner, distributor, retailer, service provider,..."
F8,Geography served,Where it sells or operates,"US regional: Alabama, Florida, Georgia and the Carolinas","one of: US regional, US national, North America, global"
F9,Company size,How large it is within its own industry,mid-sized,"one of: small, mid-sized, large, among the largest"


What the model is told for each facet:

In [9]:
print(fc.facet_guide())

F1 Core offering (offering_description): What the company sells, and to whom.
  Write two or three plain sentences: what it sells, to whom, and how it makes or delivers it.
  Example: Designs and manufactures rooftop HVAC units for commercial buildings

F2 Primary products (primary_products): The main products it sells.
  Write a short text.
  Use general names, not brand names. Name up to about five, separated by semicolons. If the company sells no products, set the status to not_applicable.
  Example: HR software; medical devices; hot-mix asphalt

F3 Primary services (primary_services): The main services it provides.
  Write a short text.
  Name up to about five, separated by semicolons. If the company sells no services, set the status to not_applicable. Support that comes with a product, such as a warranty, is not a service.
  Example: consulting; freight logistics; payroll processing

F4 Customer segment (customer_segment): Who buys its products or services.
  Choose all that apply

### 3.2 Who does what: the model and the code

Three passes (`v0/fill_facets.py`), then one check without a model:

1. **First pass, every company:** the model fills all twelve facets from what it knows, given only the company's name and website. It describes only companies it knows: one it doesn't know, can't identify, or that may be fictional comes back all unknown, never guessed from its name.
2. **Second pass, only if needed:** a company left with an unknown facet gets one more call with the pages the website scrape already saved. Only the unknown facets are taken from that answer. Nothing the model filled from its own knowledge is replaced by page text, so a poor scraped description can't bias it.
3. **Third pass, split companies only (off for now):** with splitting on, each business of a company split into several is checked against its saved pages. Since 2026-10-25 every company is one profile, so this pass doesn't run.
4. **Divestiture lines, no model:** a company whose pages mention a sale, divestiture or spin-off is flagged for the manual review.

Everything the model does is a judgment and can vary from run to run; everything code does is a fixed rule and gives the same result every time.

**The model (Claude Haiku 5.5 through the Anthropic API; or Codex CLI with `--backend codex`)**
- Reads the instructions and facet guide and the company's name and website; in the second pass also the saved page text, one labeled line at a time (`[P1.3]` is page P1, line 3).
- Says first whether it knows this specific company (`knows_company`). A name or web address alone isn't knowing it, and it must not guess from the name or from what is typical for similar names.
- For each facet, writes the status and the value, or picks the categories and writes a short detail.
- Second pass only: lists the labels of the 1-2 page lines each facet it fills relies on.
- Third pass only: says for each business whether the pages show it, citing lines, using only the pages.
- Writes a note on anything uncertain or missing.

**The code (`v0/facets.py`, `v0/fill_facets.py`)**
- **Builds the prompt:** the company's name and website; in the second pass, the page text with repeated menus and footers removed, cut at 7,000 words, each line labeled. Never any SIC code, ticker or financial field. A page line with a financial figure, a headcount, a size or growth figure, or a ticker tag is left out, using the website scrape's own filters (the scrape already dropped such sentences from the page text; this also covers page titles).
- **Checks the input:** every prompt is scanned again with the same filters, and anything found is recorded with the call (`input_figures`) and as the issue `figures_in_input`. On 2026-10-24 none of the 533 prompts, with or without pages, had one. This shows what the model was given, not what it remembers: no check can show that the model isn't drawing on figures it learned in training, which is why the output checks below exist.
- **Fixes the answer's shape:** the JSON schema goes with the prompt (Claude: `output_config.format`; Codex: `--output-schema`), which holds the model to the fields and the category lists. Company size is a category with no text, so it can't carry a figure. Code then validates the answer against the same schema; a category outside its list or a missing field makes the whole answer `invalid_answer`.
- **Drops guesses:** in the first pass, if the model says it doesn't know the company, every facet is set to unknown, whatever it wrote (issue `company_not_known`). In checks with made-up companies, version h had invented most facets from the name; version i returns them all unknown and still fills a well-known company (`facet_fill_runs.ipynb`, section 5).
- **Decides the second pass:** after the first pass, it picks the companies with an unknown facet and saved page text, and after the second call it copies over only the facets that were unknown (`facets.fill_gaps`; businesses matched B1 with B1; a company the model didn't know takes the pages answer whole). What was filled is recorded in the profile (`gap_fill`) and in `facets.csv` (`filled_from_pages`).
- **Decides what the third pass removes:** a business counts as shown only with a cited line that exists; one not shown is removed only if another is shown and none of its own name words is in the pages. If its name words are there, it is kept and flagged `split_disputed`; if the pages show no business, nothing changes (`split_not_checked`). Removed businesses stay in the profile (`split_check`).
- **Flags divestiture lines:** page lines about a sale, divestiture or spin-off (a fixed pattern) flag the company `pages_mention_divestiture`.
- **Looks up every cited label** in the lines it put in the prompt and records that line's text and page URL. A label that isn't there (say `P1.85` when page P1 stopped at line 40) is flagged `quote_not_found`. This is a lookup, not a judgment: no model is involved.
- **Records the source:** first pass, "model knowledge" for every facet that isn't unknown; second pass, "model knowledge" and/or the URLs of the cited pages.
- **Flags, never drops:** `no_value` (filled but empty), `no_source`, `website_without_quote` (website source with no line cited), `quote_not_found`, `financial_in_value` (a financial figure, headcount or ticker in any value). A flag marks the answer; it doesn't change it.
- **Caches what every call shares:** with Claude, the instructions and the answer schema (about 5,800 tokens) are sent as a cached system prompt; after the first call each company pays full price for ~40 input tokens and a tenth for the rest.
- **Keeps everything:** each call's prompt, raw answer and log under `codex_runs/` (for Claude also the whole API response, with its token counts); replaced profiles and tables move to history folders.

**Neither** checks whether a value is right; that is the manual review in the recipe's evals section.

The first-pass prompt for one company (version i: one-line JSON, details of about twenty words, lists of up to about five items, notes only for real gaps), and the page lines a second pass would add:

In [10]:
from fill_facets import company_record

EXAMPLE = "PCTY"
ex = universe[universe["target_ciq_ticker"] == EXAMPLE].iloc[0]
record = company_record(ex["IQ_COMPANY_ID"], ex["target_company_name"], ex["target_ciq_website"], with_pages=True)
prompt, _ = fc.build_prompt(record, with_pages=False)
print(f"first pass: {len(prompt.split())} words; what it says about the company:\n")
print(prompt[len(fc.instructions(False)):].strip())
_, lines = fc.build_prompt(record, with_pages=True)
print(f"\nsecond pass, only if a facet is left unknown: {len(lines)} labeled page lines, starting")
for label, text in list(lines.items())[:4]:
    print(f"  [{label}] {text[:110]}")

first pass: 1163 words; what it says about the company:

Company: Paylocity Holding Corporation
Website: www.paylocity.com

second pass, only if a facet is left unknown: 70 labeled page lines, starting
  [P1.1] Unify HR, Finance, & IT | HR & Payroll Simplified | Paylocity
  [P1.2] Paylocity unifies HR, Finance, and IT on one HCM platform, so you can stop piecing systems together and start 
  [P1.3] The next evolution of work is here to automate manual processes, reduce risk, and help you move faster.
  [P1.4] ONE experience can change everything.


### 3.3 The filled facets

`python v0/fill_facets.py` runs all three passes on the 533 universe companies and writes `facets/tables/facets.csv` (one row per business profile: for each facet `<facet>_status`, `<facet>`, `<facet>_category`, `<facet>_source`, `<facet>_flags`, and `filled_from_pages`: the facets the second pass filled) and `facets/tables/facet_quotes.csv` (one row per cited page line). Only profiles at the current prompt version count; companies not run yet show as `not_run`. The tables below are empty until that run is done:

1. how many companies are filled;
2. **coverage:** share of business profiles with each facet filled, not applicable or unknown;
3. **code checks:** share of cited labels found in the prompt, and the flags per facet;
4. **category counts** for the category facets (an `any of` facet can have several per company);
5. **the manual review list:** every profile `facets.csv` marks `needs_review`, with its `review_reasons`, all decided by code from what the fill recorded: `described_from_pages_only` (the model didn't know the company, so every facet comes from its saved pages), `facet_unknown`, `facet_flagged`, and other issues such as `pages_mention_divestiture` (its pages mention a sale or spin-off, so its facets may describe something it no longer owns);
6. **profiles:** every universe company has one profile per business, since each is a target in turn and a candidate comp for the others. They are numbered (`#`) with the company (`target`), the business, and the fill rate (share of the twelve facets filled or not applicable); the list shows the profiles with an unknown facet, then all twelve facets for a few example companies. **These examples are not a sample or a result:** they are picked by a fixed rule, the first company for each review reason and the first with nothing to review, so the cell also previews the review list. Until 2026-10-25 the cell showed three fixed tickers (PCTY, ROAD, ADUS), kept from the fill test, where Paylocity had gaps under the website-only design and the other two were ordinary picks. `PROFILES` takes any universe tickers.

In [11]:
tables = output_paths(FACETS_OUT).tables
table = pd.read_csv(tables / "facets.csv", dtype=str).fillna("")
quotes = pd.read_csv(tables / "facet_quotes.csv") if (tables / "facet_quotes.csv").exists() else pd.DataFrame(
    columns=["IQ_COMPANY_ID", "quote_found"])
# only profiles made with the current facets; older ones are redone by the next fill
ran = table[(table["prompt_version"] == fc.PROMPT_VERSION) & (table["llm_status"] == "ok")]
quotes = quotes[quotes["IQ_COMPANY_ID"].isin(ran["IQ_COMPANY_ID"])]
print(f"facets version {fc.PROMPT_VERSION}: {ran['IQ_COMPANY_ID'].nunique()} of {table['IQ_COMPANY_ID'].nunique()} universe companies filled, {len(ran)} business profiles")

facets version 2026-10-25k: 533 of 533 universe companies filled, 533 business profiles


In [12]:
if ran.empty:
    print("no companies filled with the current facets yet")
else:
    coverage = pd.DataFrame({f.name: ran[f"{f.name}_status"].value_counts(normalize=True)
                             for f in fc.FACETS}).T.rename(index=lambda n: f"{fc.FACET_ID[n]} {n}").fillna(0)
    coverage = coverage.reindex(columns=["filled", "not_applicable", "unknown"], fill_value=0)
    display((coverage * 100).round(1))

,filled,not_applicable,unknown
F1 offering_description,100.0,0.0,0.0
F2 primary_products,83.5,16.5,0.0
F3 primary_services,77.7,22.3,0.0
F4 customer_segment,100.0,0.0,0.0
F5 end_markets,100.0,0.0,0.0
F6 revenue_model,100.0,0.0,0.0
F7 value_chain_position,100.0,0.0,0.0
F8 geography_served,99.8,0.0,0.2
F9 company_size,99.6,0.0,0.4
F10 specialization,100.0,0.0,0.0


In [13]:
if not ran.empty:
    if len(quotes):
        print(f"cited page lines: {len(quotes)}, found in the prompt: {quotes['quote_found'].mean():.1%}")
    flags = pd.DataFrame({f.name: ran[f"{f.name}_flags"].str.split("; ").explode().replace("", pd.NA).value_counts()
                          for f in fc.FACETS}).T.rename(index=lambda n: f"{fc.FACET_ID[n]} {n}").fillna(0).astype(int)
    display(flags) if not flags.empty else print("no flags")

cited page lines: 330, found in the prompt: 100.0%


,website_without_quote
F1 offering_description,0
F2 primary_products,0
F3 primary_services,0
F4 customer_segment,0
F5 end_markets,0
F6 revenue_model,0
F7 value_chain_position,0
F8 geography_served,0
F9 company_size,0
F10 specialization,0


In [14]:
for f in fc.FACETS if not ran.empty else []:
    if f.kind != "text":
        counts = ran[f"{f.name}_category"].str.split("; ").explode().replace("", pd.NA).value_counts()
        print(f"{f.name}: " + ", ".join(f"{k} {v}" for k, v in counts.items()))

customer_segment: B2B 473, B2C 221, B2G 130
revenue_model: service fees 312, product sales 296, subscription 122, transaction or usage fees 88, project or contract 86, commissions or management fees 50, licensing or royalties 49, rental or leasing 46, advertising 35, interest or premiums 33, other 26, healthcare reimbursement 25
value_chain_position: service provider 321, OEM / manufacturer 183, software / platform 147, brand owner 117, asset operator 106, financial intermediary 65, distributor 35, retailer 34, component supplier 17, other 13, raw materials 10
geography_served: global 272, US national 155, US regional 54, North America 51
company_size: large 276, mid-sized 163, among the largest 73, small 19
specialization: related lines 387, niche specialist 102, diversified 44
distribution_channels: direct sales 452, online 213, own locations 161, partners / resellers 151, distributors / dealers 133, agents / brokers 56, third-party retail 50, other 25
regulatory_exposure: medium 336

In [15]:
review = ran[ran["needs_review"] == "True"].merge(universe[["IQ_COMPANY_ID", "target_ciq_ticker"]], on="IQ_COMPANY_ID")
print(f"{review['IQ_COMPANY_ID'].nunique()} of {ran['IQ_COMPANY_ID'].nunique()} companies need the manual review")
display(review["review_reasons"].str.split("; ").explode().value_counts().to_frame("profiles"))
review.set_index("target_company_name")[["target_ciq_ticker", "business_id", "review_reasons", "notes"]]

32 of 533 companies need the manual review


,profiles
review_reasons,
described_from_pages_only,19
pages_mention_divestiture,13
facet_unknown,6
facet_flagged,1


,target_ciq_ticker,business_id,review_reasons,notes
target_company_name,,,,
Acadian Asset Management Inc.,AAMI,B1,described_from_pages_only,"The given website www.bsig.com does not match the company name, so I cannot confirm which company this is and have l..."
"Accendra Health, Inc.",ACH,B1,described_from_pages_only,"I do not recognize this company from its name and website, so I could not describe it."
AdaptHealth Corp.,AHCO,B1,pages_mention_divestiture,
AMC Global Media Inc.,AMCX,B1,described_from_pages_only,"The name AMC Global Media Inc. does not match a company I recognize, and the website points to AMC Networks, so I ca..."
"American Financial Group, Inc.",AFG,B1,pages_mention_divestiture,Business mix and product details are based on general knowledge and may have changed.
Andersen Group Inc.,ANDG,B1,described_from_pages_only,The name and website alone do not identify a company I know well enough to describe currently.
Ategrity Specialty Insurance Company Holdings,ASIC,B1,described_from_pages_only; facet_unknown,"I do not know this company well enough to describe its business, so all facets are unknown."
Bioventus Inc.,BVS,B1,pages_mention_divestiture,Company size and exact product mix may have changed; details reflect my general knowledge.
Cardinal Infrastructure Group Inc.,CDNL,B1,described_from_pages_only,"I do not recognize this company from its name and website alone, so all facets are unknown."


In [16]:
# Examples to read in full, by a fixed rule (not a sample): the first company
# for each review reason, then the first with nothing to review. Any universe
# tickers can be put in PROFILES instead.
ticker = universe.set_index("IQ_COMPANY_ID")["target_ciq_ticker"]
by_reason = (ran[ran["needs_review"] == "True"].assign(reason=lambda d: d["review_reasons"].str.split("; "))
             .explode("reason").groupby("reason")["IQ_COMPANY_ID"].first())
clean = ran.loc[ran["needs_review"] == "False", "IQ_COMPANY_ID"]
PROFILES = list(dict.fromkeys([*ticker.loc[by_reason].tolist(), *ticker.loc[clean.iloc[:1]].tolist()]))

def fill_rate(rows: pd.DataFrame) -> pd.Series:
    """Share of the twelve facets filled or not applicable, per business profile."""
    return pd.concat([rows[f"{f.name}_status"].ne("unknown") for f in fc.FACETS], axis=1).mean(axis=1)

def profile(row: pd.Series) -> pd.DataFrame:
    return pd.DataFrame([{"ID": fc.FACET_ID[f.name], "facet": f.label, "status": row[f"{f.name}_status"],
                          "value": row[f.name], "source": row[f"{f.name}_source"]}
                         for f in fc.FACETS]).set_index("ID")

# every business profile, numbered: each universe company is a target in turn and a candidate comp for the others
filled = ran.merge(universe[["IQ_COMPANY_ID", "target_ciq_ticker"]], on="IQ_COMPANY_ID").reset_index(drop=True)
profiles = pd.DataFrame({"target": filled["target_company_name"] + " (" + filled["target_ciq_ticker"] + ")",
                         "business": filled["business_id"] + ": " + filled["business"],
                         "fill %": (fill_rate(filled) * 100).round(1)})
profiles.index = pd.RangeIndex(1, len(profiles) + 1, name="#")
if not profiles.empty:
    print(f"fill rate: {fill_rate(filled).mean():.1%} of facets over {len(profiles)} business profiles; "
          f"{(profiles['fill %'] == 100).sum()} profiles complete. Profiles with an unknown facet:")
    display(profiles[profiles["fill %"] < 100])

for t in PROFILES:
    name = universe.loc[universe["target_ciq_ticker"] == t, "target_company_name"].iloc[0]
    rows = profiles[filled["target_ciq_ticker"].eq(t).to_numpy()]
    if rows.empty:
        print(f"{name} ({t}): not filled with the current facets yet")
    for i in rows.index:
        reasons = filled.loc[i - 1, "review_reasons"]
        print(f"{name} ({t}): {'review: ' + reasons if reasons else 'nothing to review'}")
        display(rows.loc[[i]])
        display(profile(filled.loc[i - 1]))

fill rate: 99.9% of facets over 533 business profiles; 527 profiles complete. Profiles with an unknown facet:


,target,business,fill %
#,,,
42,Ategrity Specialty Insurance Company Holdings (ASIC),B1: specialty property and casualty insurance,83.3
219,Golf Entertainment Group Inc. (GLFE),B1: golf entertainment venues and golf courses,91.7
297,"Lumexa Imaging Holdings, Inc. (LMRI)","B1: Lumexa Imaging Holdings, Inc.",91.7
339,NPK International Inc. (NPKI),B1: temporary site access matting,91.7
389,"Pursuit Attractions and Hospitality, Inc. (PRSU)","B1: Pursuit Attractions and Hospitality, Inc.",91.7
485,"TIC Solutions, Inc. (TIC)",B1: asset integrity testing and engineering services,91.7


Acadian Asset Management Inc. (AAMI): review: described_from_pages_only


,target,business,fill %
#,,,
2,Acadian Asset Management Inc. (AAMI),B1: Acadian Asset Management Inc.,100.0


,facet,status,value,source
ID,,,,
F1,Core offering,filled,"Manages systematic equity, fixed income and alternative investment strategies for institutional investors worldwide....",model knowledge
F2,Primary products,filled,systematic equity strategies; fixed income strategies; alternative strategies,https://www.acadian-inc.com/home/default.aspx
F3,Primary services,filled,investment management; portfolio management for institutional clients,https://www.acadian-inc.com/home/default.aspx
F4,Customer segment,filled,"B2B: institutional investors such as pension funds, endowments and foundations",https://www.acadian-inc.com/home/default.aspx
F5,End markets,filled,investment management; asset management; financial services,model knowledge
F6,Revenue model,filled,"commissions or management fees: asset-based management fees on client portfolios, plus performance fees on some stra...",model knowledge
F7,Value chain position,filled,service provider: runs investment portfolios for institutional clients and is paid by them,model knowledge
F8,Geography served,filled,"global: headquartered in Boston with affiliates in London, Singapore and Sydney, clients worldwide",https://www.acadian-inc.com/home/default.aspx
F9,Company size,filled,mid-sized,model knowledge


...

,target,business,fill %
#,,,
7,AdaptHealth Corp. (AHCO),B1: AdaptHealth Corp.,100.0


,facet,status,value,source
ID,,,,
F1,Core offering,filled,"Provides home medical equipment, respiratory therapy, diabetes supplies and pharmacy services to patients in the Uni...",model knowledge
F2,Primary products,filled,respiratory devices and CPAP equipment; home oxygen equipment; mobility equipment; diabetes testing and insulin supp...,model knowledge
F3,Primary services,filled,home respiratory therapy; sleep therapy support; care management; pharmacy services,model knowledge
F4,Customer segment,filled,B2C: patients with chronic conditions receiving care at home; B2G: Medicare and Medicaid programs as payers; B2B: co...,model knowledge
F5,End markets,filled,healthcare; home healthcare; pharmacy; diabetes care,model knowledge
F6,Revenue model,filled,healthcare reimbursement: government and commercial payers reimburse equipment and supplies provided to patients; re...,model knowledge
F7,Value chain position,filled,"service provider: delivers home care and therapy to patients, billed to payers; distributor: supplies equipment and ...",model knowledge
F8,Geography served,filled,US national: Operates across many US states through a national network of locations,model knowledge
F9,Company size,filled,large,model knowledge


AAR Corp. (AIR): nothing to review


,target,business,fill %
#,,,
1,AAR Corp. (AIR),B1: AAR Corp.,100.0


,facet,status,value,source
ID,,,,
F1,Core offering,filled,"Supplies aircraft parts, repair and overhaul services, and supply chain solutions to commercial airlines, cargo oper...",model knowledge
F2,Primary products,filled,aircraft components; landing gear; expeditionary shelters; military containers,model knowledge
F3,Primary services,filled,aircraft maintenance and repair; component overhaul; integrated supply chain management; aircraft modification,model knowledge
F4,Customer segment,filled,"B2B: airlines, cargo carriers, aircraft lessors and maintenance providers; B2G: US and allied military and governmen...",model knowledge
F5,End markets,filled,commercial aviation; defense and government; air cargo,model knowledge
F6,Revenue model,filled,"product sales: aircraft parts and military shelters sold to customers; service fees: repair, overhaul and maintenanc...",model knowledge
F7,Value chain position,filled,"service provider: maintenance, repair and supply chain services sold to airlines and operators; OEM / manufacturer: ...",model knowledge
F8,Geography served,filled,"global: customers worldwide, with repair facilities in the US and several other countries",model knowledge
F9,Company size,filled,large,model knowledge
